In [ ]:

# IMPORTS


import sqlite3
import os
import re
import json
import uuid
import math
import datetime
from pathlib import Path

# NLP
import nltk
import spacy

# Data Processing
import pandas as pd
import numpy as np

# Embeddings
from sentence_transformers import SentenceTransformer

# Similarity Search
import faiss

# Graph Processing
import networkx as nx

# Optional ML Utilities
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Collections
from collections import Counter, defaultdict

# Typing
from typing import List, Dict, Tuple, Any


# DATABASE INITIALIZATION


DATABASE_PATH = "personal_memory.db"


def initialize_database():
    """
    Create all database tables if they do not exist.
    """

    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()


    # NOTES
 

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS notes (
        note_id INTEGER PRIMARY KEY AUTOINCREMENT,
        content TEXT,
        summary TEXT,
        importance_score REAL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    # MEMORIES


    cursor.execute("""
    CREATE TABLE IF NOT EXISTS memories (
        memory_id INTEGER PRIMARY KEY AUTOINCREMENT,
        memory_text TEXT,
        memory_type TEXT,
        confidence REAL,
        source_note_id INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    # TOPICS


    cursor.execute("""
    CREATE TABLE IF NOT EXISTS topics (
        topic_id INTEGER PRIMARY KEY AUTOINCREMENT,
        topic_name TEXT,
        frequency INTEGER DEFAULT 0
    )
    """)

 
    # ENTITIES
  

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS entities (
        entity_id INTEGER PRIMARY KEY AUTOINCREMENT,
        entity_name TEXT,
        entity_type TEXT
    )
    """)

    # RELATIONSHIPS
    
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS relationships (
        relationship_id INTEGER PRIMARY KEY AUTOINCREMENT,
        source_node TEXT,
        relation TEXT,
        target_node TEXT
    )
    """)

    # EMBEDDINGS
 

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS embeddings (
        embedding_id INTEGER PRIMARY KEY AUTOINCREMENT,
        note_id INTEGER,
        vector BLOB
    )
    """)

    # OBSERVATIONS


    cursor.execute("""
    CREATE TABLE IF NOT EXISTS observations (
        observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
        observation_text TEXT
    )
    """)

    # PREDICTIONS
  

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS predictions (
        prediction_id INTEGER PRIMARY KEY AUTOINCREMENT,
        prediction_text TEXT
    )
    """)

    conn.commit()
    conn.close()

    print("Database initialized successfully.")

# MEMORY INITIALIZATION


class MemorySystem:
    """
    Core in-memory structures used by the AI system.
    """

    def __init__(self):

        # Knowledge Graph
        self.knowledge_graph = nx.DiGraph()

        # Timeline Events
        self.timeline = []

        # Vector Search Index
        self.vector_index = None

        # Extracted Information
        self.beliefs = []
        self.goals = []
        self.projects = []

        # Knowledge Collections
        self.topics = []
        self.entities = []

        # Fast Memory Cache
        self.memory_cache = {}

        # Statistics
        self.total_notes = 0
        self.total_memories = 0

    def initialize_vector_index(self, embedding_dimension=384):
        """
        Create FAISS vector index.
        """

        self.vector_index = faiss.IndexFlatL2(
            embedding_dimension
        )

        print(
            f"Vector Index Initialized "
            f"(Dimension={embedding_dimension})"
        )

    def display_status(self):

        print("\nMemory System Initialized")

        print(f"Notes: {self.total_notes}")
        print(f"Goals: {len(self.goals)}")
        print(f"Projects: {len(self.projects)}")
        print(f"Beliefs: {len(self.beliefs)}")
        print(f"Topics: {len(self.topics)}")
        print(f"Entities: {len(self.entities)}")


# Create Global Memory System

memory_system = MemorySystem()

memory_system.initialize_vector_index()


# NOTES IMPORT SYSTEM


NOTES_FOLDER = "notes"


def import_notes(notes_folder=NOTES_FOLDER):
    """
    Load all txt notes from folder.
    """

    notes_collection = []

    folder_path = Path(notes_folder)

    if not folder_path.exists():

        print(f"Folder not found: {notes_folder}")

        return notes_collection

    txt_files = list(
        folder_path.rglob("*.txt")
    )

    print(
        f"Found {len(txt_files)} text files."
    )

    for file_path in txt_files:

        try:

            with open(
                file_path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as file:

                content = file.read()

            content = content.encode(
                "utf-8",
                errors="ignore"
            ).decode("utf-8")

            content = re.sub(
                r"[\x00-\x1F\x7F]",
                " ",
                content
            )

            note_data = {
                "file_name": file_path.name,
                "file_path": str(file_path),
                "content": content,
                "length": len(content),
                "import_time":
                    datetime.datetime.now()
                    .isoformat()
            }

            notes_collection.append(
                note_data
            )

            print(
                f"Imported: {file_path.name}"
            )

        except Exception as error:

            print(
                f"Failed: {file_path.name}"
            )

            print(error)

    print(
        f"\nSuccessfully Imported "
        f"{len(notes_collection)} Notes"
    )

    return notes_collection

